# ASR Speech Benchmark for PoliMillionaire

This notebook benchmarks speech recognition only. It does not load V5 indexes, rerankers, or LLMs.

Workflow:
1. Start speech-mode games.
2. Fetch question + A/B/C/D WAV files.
3. Submit a random answer quickly, only to advance/close the game.
4. Transcribe the saved WAV clips offline with selected ASR models.
5. Log one CSV row per clip/model with latency and transcript.


In [ ]:
# Minimal dependencies for API + audio + ASR benchmarking.
import os, sys, json, time, random, gc, math, io, re
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = os.path.exists('/content') and not os.path.exists('/kaggle')
IN_KAGGLE = os.path.exists('/kaggle')
WORK_ROOT = Path('/kaggle/working') if IN_KAGGLE else (Path('/content/asr_benchmark_runtime') if IN_COLAB else Path.cwd() / '.runtime')
LOG_DIR = WORK_ROOT / 'logs'
AUDIO_ROOT = WORK_ROOT / 'asr_audio'
LOG_DIR.mkdir(parents=True, exist_ok=True)
AUDIO_ROOT.mkdir(parents=True, exist_ok=True)

API_URL = 'http://131.175.15.22:51111/'
USERNAME_SECRET_NAME = 'USERNAME'
PASSWORD_SECRET_NAME = 'PASSWORD'
USERNAME = None
PASSWORD = None

ASR_BENCH_RUN_ID = time.strftime('%Y%m%d_%H%M%S_asr')
AUDIO_RUN_DIR = AUDIO_ROOT / ASR_BENCH_RUN_ID
MANIFEST_CSV = LOG_DIR / 'asr_audio_manifest.csv'
TRANSCRIPT_CSV = LOG_DIR / 'asr_benchmark_transcripts.csv'

RANDOM_SEED = 42
random.seed(RANDOM_SEED)

print('WORK_ROOT:', WORK_ROOT)
print('AUDIO_RUN_DIR:', AUDIO_RUN_DIR)
print('MANIFEST_CSV:', MANIFEST_CSV)
print('TRANSCRIPT_CSV:', TRANSCRIPT_CSV)


In [ ]:
# Locate and import the official PoliMillionaire client.
def add_api_client_to_path():
    candidates = []
    if IN_KAGGLE:
        roots = [Path('/kaggle/input')]
        for root in roots:
            if root.exists():
                candidates.extend(root.rglob('NLP_assignment_api_client'))
    candidates.extend([
        Path.cwd() / 'api_client' / 'NLP_assignment_api_client',
        Path.cwd().parents[0] / 'api_client' / 'NLP_assignment_api_client' if len(Path.cwd().parents) > 0 else Path(''),
    ])
    for p in candidates:
        if p and (p / 'millionaire_client').exists():
            if str(p) not in sys.path:
                sys.path.insert(0, str(p))
            print('Using API client:', p)
            return p
    raise FileNotFoundError('Could not find NLP_assignment_api_client/millionaire_client.')

API_CLIENT_DIR = add_api_client_to_path()

from millionaire_client import MillionaireClient

def read_secret(secret_name):
    if not secret_name:
        return None
    if secret_name in os.environ and os.environ[secret_name]:
        return os.environ[secret_name]
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(secret_name)
    except Exception:
        return None

def setup_client():
    username = USERNAME or read_secret(USERNAME_SECRET_NAME)
    password = PASSWORD or read_secret(PASSWORD_SECRET_NAME)
    if not username or not password:
        raise ValueError('Set USERNAME/PASSWORD or Kaggle secrets USERNAME/PASSWORD.')
    client = MillionaireClient(API_URL)
    client.login(username, password)
    return client

def get_competition_names(client):
    comps = client.competitions.list_all()
    for comp in comps:
        print(comp.id, comp.name, getattr(comp, 'max_levels', None))
    return {comp.id: comp.name for comp in comps}

client = setup_client()
competition_names = get_competition_names(client)


In [ ]:
# Audio collection settings. Random answers are submitted before ASR, so all models see the same saved WAV files.
COLLECT_COMPETITIONS = {'Ancient History and Politics'}
SESSIONS_PER_COMPETITION = 10
MAX_QUESTIONS_PER_SESSION = 15
WAIT_BETWEEN_SESSIONS_SEC = 2.0
SUBMIT_RANDOM_ANSWER = True

def append_csv_row(row, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df = pd.DataFrame([row])
    df.to_csv(path, mode='a', header=not path.exists() or path.stat().st_size == 0, index=False)

def option_records(question):
    return [(int(opt.id), getattr(opt, 'text', None)) for opt in (getattr(question, 'options', []) or [])]

def save_bytes(path, data):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_bytes(data)
    return str(path)

def fetch_current_question_audio(game, comp_id, comp_name, attempt_number, question_index):
    question = game.current_question
    if question is None:
        return None

    session_id = getattr(game, 'session_id', None)
    level = getattr(question, 'level', None)
    qid = getattr(question, 'id', None)
    base = AUDIO_RUN_DIR / f'comp_{comp_id}' / f'attempt_{attempt_number:03d}' / f'session_{session_id}_q{question_index:02d}_qid_{qid}'

    clip_paths = {}
    fetch_times = {}
    clip_sizes = {}

    started = time.time()
    q_audio = game.fetch_audio_question()
    fetch_times['question'] = time.time() - started
    clip_paths['question'] = save_bytes(base / 'question.wav', q_audio)
    clip_sizes['question'] = len(q_audio)

    for idx in range(4):
        label = f'option_{chr(65 + idx)}'
        started = time.time()
        audio = game.fetch_audio_option_next()
        fetch_times[label] = time.time() - started
        clip_paths[label] = save_bytes(base / f'{label}.wav', audio)
        clip_sizes[label] = len(audio)

    game.refresh_state()
    time_remaining_after_audio = getattr(game, 'time_remaining', None)

    chosen_option_id = None
    answer_error = None
    result_fields = {}
    if SUBMIT_RANDOM_ANSWER:
        options = list(getattr(question, 'options', []) or [])
        if options:
            chosen_option_id = int(random.choice(options).id)
            try:
                answer_started = time.time()
                result = game.answer(chosen_option_id)
                result_fields = {
                    'answer_latency_seconds': time.time() - answer_started,
                    'random_answer_correct': getattr(result, 'correct', None),
                    'random_answer_timed_out': getattr(result, 'timed_out', None),
                    'random_answer_game_over': getattr(result, 'game_over', None),
                    'earned_amount': getattr(result, 'earned_amount', None),
                }
            except Exception as exc:
                answer_error = repr(exc)

    row = {
        'audio_run_id': ASR_BENCH_RUN_ID,
        'attempt_number': attempt_number,
        'question_index': question_index,
        'session_id': session_id,
        'competition_id': comp_id,
        'competition_name': comp_name,
        'question_id': qid,
        'question_level': level,
        'api_question_text': getattr(question, 'text', None),
        'api_options_json': json.dumps(option_records(question), ensure_ascii=False),
        'clip_paths_json': json.dumps(clip_paths, ensure_ascii=False),
        'fetch_times_json': json.dumps(fetch_times, ensure_ascii=False),
        'clip_sizes_json': json.dumps(clip_sizes, ensure_ascii=False),
        'time_remaining_after_audio': time_remaining_after_audio,
        'random_option_id': chosen_option_id,
        'answer_error': answer_error,
        **result_fields,
    }
    append_csv_row(row, MANIFEST_CSV)
    return row

def collect_audio_dataset(client, competition_names):
    rows = []
    for comp_id, comp_name in competition_names.items():
        if comp_name not in COLLECT_COMPETITIONS:
            continue
        for attempt in range(1, SESSIONS_PER_COMPETITION + 1):
            print(f'\n=== Collect attempt {attempt}/{SESSIONS_PER_COMPETITION} | {comp_name} ===', flush=True)
            try:
                game = client.game.start(competition_id=comp_id, mode='speech')
            except Exception as exc:
                row = {
                    'audio_run_id': ASR_BENCH_RUN_ID,
                    'attempt_number': attempt,
                    'competition_id': comp_id,
                    'competition_name': comp_name,
                    'start_error': repr(exc),
                }
                append_csv_row(row, MANIFEST_CSV)
                rows.append(row)
                continue

            question_index = 0
            while getattr(game, 'in_progress', False) and question_index < MAX_QUESTIONS_PER_SESSION:
                question_index += 1
                try:
                    row = fetch_current_question_audio(game, comp_id, comp_name, attempt, question_index)
                    if row is None:
                        break
                    rows.append(row)
                    print(f"qid={row.get('question_id')} level={row.get('question_level')} timer={row.get('time_remaining_after_audio')} random_correct={row.get('random_answer_correct')}", flush=True)
                except Exception as exc:
                    row = {
                        'audio_run_id': ASR_BENCH_RUN_ID,
                        'attempt_number': attempt,
                        'question_index': question_index,
                        'session_id': getattr(game, 'session_id', None),
                        'competition_id': comp_id,
                        'competition_name': comp_name,
                        'collection_error': repr(exc),
                    }
                    append_csv_row(row, MANIFEST_CSV)
                    rows.append(row)
                    break
            time.sleep(WAIT_BETWEEN_SESSIONS_SEC)
    return pd.DataFrame(rows)


In [ ]:
# Run this cell to collect a fresh audio dataset from speech mode.
# Keep SESSIONS_PER_COMPETITION small while testing.
manifest_df = collect_audio_dataset(client, competition_names)
manifest_df.tail()


In [ ]:
# ASR model configs. Enable one or more models for the benchmark pass.
ASR_DEVICE = 'cuda:1'
ASR_DTYPE = 'float16'
ASR_PROMPT = (
    'Multiple choice quiz. Terms may include Achaeans, Argives, Danaans, Panhellenes, '
    'Ahhiyawa, Tawagalawa, Wilusa, Madduwatta, Ephors, Sparta, Carthage, '
    'Antikythera, Consecratio, Augurium, Acropolis, Illyrians, Pelasgians.'
)

ASR_MODEL_CONFIGS = [
    {
        'key': 'distil_medium_en',
        'model_id': 'distil-whisper/distil-medium.en',
        'backend': 'whisper_seq2seq',
        'enabled': True,
        'device': ASR_DEVICE,
        'dtype': ASR_DTYPE,
        'max_length': 96,
        'num_beams': 1,
        'prompt': ASR_PROMPT,
    },
    {
        'key': 'whisper_large_v3_turbo',
        'model_id': 'openai/whisper-large-v3-turbo',
        'backend': 'whisper_seq2seq',
        'enabled': False,
        'device': ASR_DEVICE,
        'dtype': ASR_DTYPE,
        'max_length': 128,
        'num_beams': 3,
        'prompt': ASR_PROMPT,
    },
    {
        'key': 'parakeet_tdt_0_6b_v3',
        'model_id': 'nvidia/parakeet-tdt-0.6b-v3',
        'backend': 'hf_asr_pipeline',
        'enabled': False,
        'device': ASR_DEVICE,
        'dtype': ASR_DTYPE,
        'install_transformers_from_source': True,
    },
    {
        'key': 'qwen3_asr_1_7b',
        'model_id': 'Qwen/Qwen3-ASR-1.7B',
        'backend': 'qwen_asr',
        'enabled': False,
        'device': ASR_DEVICE,
        'dtype': 'bfloat16',
        'max_new_tokens': 256,
    },
    {
        'key': 'whisper_medium_en',
        'model_id': 'openai/whisper-medium.en',
        'backend': 'whisper_seq2seq',
        'enabled': False,
        'device': ASR_DEVICE,
        'dtype': ASR_DTYPE,
        'max_length': 96,
        'num_beams': 3,
        'prompt': ASR_PROMPT,
    },
]


In [ ]:
# ASR backend implementations.
def install_base_asr_deps():
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers', 'accelerate', 'soundfile', 'scipy'])

def maybe_install_transformers_source():
    import subprocess
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'git+https://github.com/huggingface/transformers'])

def decode_wav(path, target_sr=16000):
    import soundfile as sf
    from scipy.signal import resample_poly
    audio, sr = sf.read(str(path), dtype='float32', always_2d=False)
    if getattr(audio, 'ndim', 1) > 1:
        audio = audio.mean(axis=1)
    audio = np.asarray(audio, dtype=np.float32)
    if sr != target_sr:
        g = math.gcd(int(sr), int(target_sr))
        audio = resample_poly(audio, target_sr // g, int(sr) // g).astype(np.float32)
        sr = target_sr
    return audio, int(sr)

def audio_duration_seconds(path):
    import soundfile as sf
    info = sf.info(str(path))
    return float(info.frames) / float(info.samplerate) if info.samplerate else None

def clean_transcript(text):
    text = str(text or '').strip()
    text = re.sub(r'\s+', ' ', text)
    return text

def dtype_from_name(name):
    import torch
    if str(name).lower() in ('float16', 'fp16'):
        return torch.float16
    if str(name).lower() in ('bfloat16', 'bf16'):
        return torch.bfloat16
    return torch.float32

class WhisperSeq2SeqRunner:
    def __init__(self, cfg):
        install_base_asr_deps()
        import torch
        from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
        self.cfg = cfg
        self.device = torch.device(cfg.get('device', 'cuda:0') if torch.cuda.is_available() else 'cpu')
        self.dtype = dtype_from_name(cfg.get('dtype', 'float16')) if self.device.type == 'cuda' else torch.float32
        self.processor = AutoProcessor.from_pretrained(cfg['model_id'])
        self.model = AutoModelForSpeechSeq2Seq.from_pretrained(
            cfg['model_id'],
            torch_dtype=self.dtype,
            low_cpu_mem_usage=True,
            use_safetensors=True,
        ).to(self.device)
        self.model.eval()
        if hasattr(self.model, 'generation_config') and hasattr(self.model.generation_config, 'return_timestamps'):
            self.model.generation_config.return_timestamps = False

    def transcribe(self, path):
        import torch
        audio, sr = decode_wav(path, target_sr=16000)
        inputs = self.processor(audio, sampling_rate=sr, return_tensors='pt')
        input_features = inputs.input_features.to(device=self.device, dtype=self.dtype)
        kwargs = {
            'max_length': int(self.cfg.get('max_length', 96)),
            'return_timestamps': False,
            'num_beams': int(self.cfg.get('num_beams', 1)),
        }
        prompt = self.cfg.get('prompt')
        if prompt:
            try:
                kwargs['prompt_ids'] = self.processor.get_prompt_ids(prompt, return_tensors='pt').to(self.device)
            except Exception:
                pass
        for extra in ({'language': 'english', 'task': 'transcribe'}, {}):
            try:
                with torch.inference_mode():
                    pred = self.model.generate(input_features, **kwargs, **extra)
                return clean_transcript(self.processor.batch_decode(pred, skip_special_tokens=True)[0])
            except TypeError:
                continue
        with torch.inference_mode():
            pred = self.model.generate(input_features, **kwargs)
        return clean_transcript(self.processor.batch_decode(pred, skip_special_tokens=True)[0])

    def close(self):
        del self.model
        del self.processor

class HfAsrPipelineRunner:
    def __init__(self, cfg):
        if cfg.get('install_transformers_from_source'):
            maybe_install_transformers_source()
        else:
            install_base_asr_deps()
        import torch
        from transformers import pipeline
        self.cfg = cfg
        device = cfg.get('device', 'cuda:0')
        if torch.cuda.is_available() and str(device).startswith('cuda'):
            device_arg = int(str(device).split(':')[1]) if ':' in str(device) else 0
            dtype = dtype_from_name(cfg.get('dtype', 'float16'))
        else:
            device_arg = -1
            dtype = torch.float32
        self.pipe = pipeline('automatic-speech-recognition', model=cfg['model_id'], device=device_arg, torch_dtype=dtype)

    def transcribe(self, path):
        out = self.pipe(str(path))
        if isinstance(out, dict):
            return clean_transcript(out.get('text', ''))
        return clean_transcript(out)

    def close(self):
        del self.pipe

class QwenAsrRunner:
    def __init__(self, cfg):
        import subprocess
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'qwen-asr'])
        import torch
        from qwen_asr import Qwen3ASRModel
        self.cfg = cfg
        dtype = dtype_from_name(cfg.get('dtype', 'bfloat16'))
        self.model = Qwen3ASRModel.from_pretrained(
            cfg['model_id'],
            dtype=dtype,
            device_map=cfg.get('device', 'cuda:0'),
            max_inference_batch_size=1,
            max_new_tokens=int(cfg.get('max_new_tokens', 256)),
        )

    def transcribe(self, path):
        results = self.model.transcribe(audio=str(path), language='English')
        if isinstance(results, list) and results:
            return clean_transcript(getattr(results[0], 'text', results[0]))
        return clean_transcript(results)

    def close(self):
        del self.model

def make_runner(cfg):
    backend = cfg['backend']
    if backend == 'whisper_seq2seq':
        return WhisperSeq2SeqRunner(cfg)
    if backend == 'hf_asr_pipeline':
        return HfAsrPipelineRunner(cfg)
    if backend == 'qwen_asr':
        return QwenAsrRunner(cfg)
    raise ValueError(f'Unknown backend: {backend}')

def close_runner(runner):
    try:
        runner.close()
    finally:
        gc.collect()
        try:
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
        except Exception:
            pass


In [ ]:
# Run enabled ASR models on saved audio clips.
def load_manifest(path=MANIFEST_CSV, audio_run_id=None):
    df = pd.read_csv(path)
    df = df[df['clip_paths_json'].notna()].copy()
    if audio_run_id:
        df = df[df['audio_run_id'] == audio_run_id].copy()
    return df

def iter_manifest_clips(row):
    paths = json.loads(row['clip_paths_json'])
    for label in ['question', 'option_A', 'option_B', 'option_C', 'option_D']:
        if label in paths and paths[label]:
            yield label, Path(paths[label])

def benchmark_asr_models(manifest_df, configs, output_csv=TRANSCRIPT_CSV):
    all_rows = []
    enabled = [cfg for cfg in configs if cfg.get('enabled')]
    print('Enabled models:', [cfg['key'] for cfg in enabled])
    for cfg in enabled:
        runner = None
        load_started = time.time()
        try:
            print(f"\nLoading {cfg['key']} | {cfg['model_id']} | backend={cfg['backend']}", flush=True)
            runner = make_runner(cfg)
            load_seconds = time.time() - load_started
            print(f'Loaded in {load_seconds:.1f}s', flush=True)
            for _, row in manifest_df.iterrows():
                for clip_label, clip_path in iter_manifest_clips(row):
                    result = {
                        'audio_run_id': row.get('audio_run_id'),
                        'asr_run_id': ASR_BENCH_RUN_ID,
                        'model_key': cfg['key'],
                        'model_id': cfg['model_id'],
                        'backend': cfg['backend'],
                        'load_seconds_for_model': load_seconds,
                        'competition_id': row.get('competition_id'),
                        'competition_name': row.get('competition_name'),
                        'attempt_number': row.get('attempt_number'),
                        'question_index': row.get('question_index'),
                        'session_id': row.get('session_id'),
                        'question_id': row.get('question_id'),
                        'question_level': row.get('question_level'),
                        'clip_label': clip_label,
                        'clip_path': str(clip_path),
                        'api_question_text': row.get('api_question_text'),
                        'api_options_json': row.get('api_options_json'),
                    }
                    try:
                        result['audio_duration_seconds'] = audio_duration_seconds(clip_path)
                        started = time.time()
                        transcript = runner.transcribe(clip_path)
                        result['transcription_seconds'] = time.time() - started
                        result['transcript'] = transcript
                        result['error_message'] = None
                        print(f"{cfg['key']} {row.get('question_id')} {clip_label}: {result['transcription_seconds']:.2f}s | {transcript[:90]}", flush=True)
                    except Exception as exc:
                        result['transcription_seconds'] = None
                        result['transcript'] = None
                        result['error_message'] = repr(exc)
                        print(f"ERROR {cfg['key']} {row.get('question_id')} {clip_label}: {exc}", flush=True)
                    append_csv_row(result, output_csv)
                    all_rows.append(result)
        finally:
            if runner is not None:
                close_runner(runner)
    return pd.DataFrame(all_rows)

# By default, benchmark the current audio collection run. Change audio_run_id to reuse an older dataset.
manifest_for_benchmark = load_manifest(MANIFEST_CSV, audio_run_id=ASR_BENCH_RUN_ID)
print('Manifest rows for benchmark:', len(manifest_for_benchmark))
transcript_df = benchmark_asr_models(manifest_for_benchmark, ASR_MODEL_CONFIGS)
transcript_df.tail()


In [ ]:
# Quick summary of benchmark outputs.
if TRANSCRIPT_CSV.exists():
    df = pd.read_csv(TRANSCRIPT_CSV)
    print('Rows:', len(df))
    display(df.groupby(['model_key', 'clip_label'])['transcription_seconds'].agg(['count', 'mean', 'min', 'max']).reset_index())
    cols = ['model_key', 'competition_name', 'question_id', 'clip_label', 'transcript', 'transcription_seconds', 'error_message']
    display(df[cols].tail(40))
else:
    print('No transcript CSV yet:', TRANSCRIPT_CSV)


## Suggested Model Plan

Start with:
- `distil-whisper/distil-medium.en` as the known fast baseline.
- `openai/whisper-large-v3-turbo` as the stronger Whisper-family candidate.
- `nvidia/parakeet-tdt-0.6b-v3` as the strongest lightweight non-Whisper candidate. Its model card documents Transformers pipeline and AutoModelForTDT usage, but may require installing Transformers from source until the release reaches the Kaggle image.
- `Qwen/Qwen3-ASR-1.7B` as an optional heavier candidate via the `qwen-asr` package.

Keep `openai/whisper-medium.en` disabled unless you want an English-only Whisper comparison.
